In [ ]:
!pip install spacy
!pip install -U pyarrow

In [ ]:
import spacy
import pandas as pd
from collections import defaultdict

spacy.cli.download("en_core_web_trf")


class EntityExtractor:
    """
    A class to extract named entities (persons and organizations) from text articles using spaCy.
    """

    def __init__(self, model_name: str = "en_core_web_trf"):
        """
        Initialize the NLP pipeline.

        Args:
            model_name (str): spaCy model to load.
        """
        self.nlp = spacy.load(model_name)
        print(f"Model '{model_name}' loaded.")

    def extract_entities(self, text: str) -> dict:
        """
        Extracts PERSON and ORG entities from a single article.

        Args:
            text (str): The article text.

        Returns:
            dict: Dictionary with 'Persons' and 'Organizations' as keys and lists of entity names.
        """
        doc = self.nlp(text)
        entity_dict = defaultdict(set)

        for ent in doc.ents:
            if ent.label_ in ["PERSON", "ORG"]:
                entity_dict[ent.label_].add(ent.text)

        return {
            "Persons": list(entity_dict["PERSON"]),
            "Organizations": list(entity_dict["ORG"])
        }

    def process_articles(self, articles: list, save_checkpoint=500) -> pd.DataFrame:
        """
        Processes a list of articles and extracts named entities for each.

        Args:
            articles (list): List of article strings.

        Returns:
            pd.DataFrame: DataFrame with extracted entities per article.
        """
        all_entities = []

        for idx, article in enumerate(articles):
            print(f"Processing Article {idx + 1} of {len(articles)}")
            if not article:
                print(f"Skipping empty article at index {idx}")
                continue

            extracted = self.extract_entities(article)
            all_entities.append({
                "ArticleID": idx + 1,
                "Persons": extracted["Persons"],
                "Organizations": extracted["Organizations"]
            })

            if (idx + 1) % save_checkpoint == 0:
                self.save_to_parquet(pd.DataFrame(all_entities), f"entity_results_{idx + 1}.parquet")
                print(f"Checkpoint saved at Article {idx + 1}")

        return pd.DataFrame(all_entities)

    def save_to_parquet(self, df: pd.DataFrame, output_path: str = "entity_results.parquet") -> None:
        """
        Saves the extracted entities to a Parquet file.

        Args:
            df (pd.DataFrame): DataFrame of extracted entities.
            output_path (str): File path to save the output.
        """
        df.to_parquet(output_path, engine="pyarrow", index=False)
        print(f"Results saved to {output_path}")


# Example usage
if __name__ == "__main__":
    # Load the articles
    news_df = pd.read_parquet('cleaned_news_data.parquet', engine='pyarrow')
    articles = news_df['clean_full_text'].iloc[12500:15000].tolist()

    # Initialize the entity extractor
    extractor = EntityExtractor()

    # Process the articles
    df = extractor.process_articles(articles)

    # Save the results to Parquet format
    extractor.save_to_parquet(df, "named_entities_12500.parquet")
    print("Results saved to named_entities_12500.parquet")


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Model 'en_core_web_trf' loaded.
Processing Article 1 of 2500
Processing Article 2 of 2500
Processing Article 3 of 2500
Processing Article 4 of 2500
Processing Article 5 of 2500
Processing Article 6 of 2500
Processing Article 7 of 2500
Processing Article 8 of 2500
Processing Article 9 of 2500
Processing Article 10 of 2500
Skipping empty article at index 9
Processing Article 11 of 2500
Processing Article 12 of 2500
Processing Article 13 of 2500
Skipping empty article at index 12
Processing Article 14 of 2500
Processing Article 15 of 2500
Processing Article 16 of 2500
Processing Article 17 of 2500
Processing Article 18 of 2500
Processing Article 19 of 2500
P

In [ ]:
!pip install spacy-entity-linker

  Preparing metadata (setup.py) ... done
  Created wheel for spacy-entity-linker: filename=spacy_entity_linker-1.0.3-py3-none-any.whl size=14373 sha256=8216e099e0f7f34f5e3b07591e34ff06b930aaa59a5194364399267aaf94c515
  Stored in directory: /root/.cache/pip/wheels/fa/f5/bf/5bb90acb1b36f206c1b0c007fbf6ac1058ce0b989ee6fcc38e
Successfully built spacy-entity-linker


In [ ]:
import spacy
import pandas as pd
from tqdm import tqdm

spacy.cli.download("en_core_web_md")

class EntityLinker:
    """
    A class to link extracted named entities to knowledge base entries (e.g., Wikidata).
    """

    def __init__(self, model_name="en_core_web_md"):
        """
        Initializes spaCy model with the entity linker.

        Args:
            model_name (str): spaCy model to use.
        """
        self.nlp = spacy.load(model_name)
        self.nlp.add_pipe("entityLinker", last=True)
        self.linker = self.nlp.get_pipe("entityLinker")
        print("Entity linker loaded into the pipeline.")

    def link_entity(self, entity_text):
        """
        Links a single entity name to a KB entry.

        Args:
            entity_text (str): Raw entity string.

        Returns:
            dict: Linking result with 'name', 'url', and 'score'.
        """
        doc = self.nlp(entity_text)
        links = doc._.linkedEntities

        if links:
            top = links[0]
            # print(dir(top))
            return {
                "original": entity_text,
                "linked_name": top.label,
                "url": top.url,
                "id": top.get_id(),
            }
        else:
            return {
                "original": entity_text,
                "linked_name": None,
                "url": None,
                "id": None,
            }

    def process_entity_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Process the DataFrame returned by EntityExtractor and link entities.

        Args:
            df (pd.DataFrame): DataFrame with 'Persons' and 'Organizations' columns.

        Returns:
            pd.DataFrame: Same structure but with linked entity info added.
        """
        linked_rows = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Linking entities"):
            linked_persons = [self.link_entity(p) for p in row["Persons"]]
            linked_orgs = [self.link_entity(o) for o in row["Organizations"]]

            linked_rows.append({
                "ArticleID": row["ArticleID"],
                "LinkedPersons": linked_persons,
                "LinkedOrganizations": linked_orgs,
            })

        return pd.DataFrame(linked_rows)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
linker = EntityLinker()
# extracted_df = pd.read_parquet('named_entities.parquet', engine='pyarrow')
filename_lst = ['named_entities_0.parquet','named_entities_10000.parquet','named_entities_12500.parquet','named_entities_15000.parquet']
extracted_df = pd.concat([pd.read_parquet(filename, engine='pyarrow') for filename in filename_lst], ignore_index = True)
extracted_df.to_parquet('named_entities.parquet', engine = 'pyarrow')
linked_df = linker.process_entity_dataframe(extracted_df)
linked_df.to_parquet("linked_entities.parquet", index=False)

Entity linker loaded into the pipeline.


Linking entities: 100%|██████████| 19199/19199 [47:14<00:00,  6.77it/s]
